# Spice Net
This is a quick guide on how to use the python spice net package.

In [1]:
import numpy as np
import pandas as pd

import spice_net as spn
import sinabs


In [2]:
df = pd.read_csv('data/test_data.csv')
df

,init,result
0,-0.868245,-0.654525
1,0.866917,0.651528
2,-0.102037,-0.001062
3,0.602888,0.219135
4,0.838314,0.589141
...,...,...
2995,-0.181318,-0.005961
2996,0.314093,0.030987
2997,-0.313361,-0.030771
2998,0.144365,0.003009


## Initiation
First of all you need to create 2 self-organizing maps and 1 hebbian correlation matrix. The hardest part is figuring out the LRFs (= learning rate functions). You can find some implementations in the package, if you want to implement one for your self inherit the ```LearningRateFunction``` class. 

In [3]:
som_size = 100
som_2_size = 100

som_1 = spn.SpiceNetSom(n_neurons=som_size,
                        value_range_start=df['init'].min(),
                        value_range_end=df['init'].max(),
                        lrf_tuning_curve=spn.ConstLRF(0.7),
                        lrf_interaction_kernel=spn.ConstLRF(0.7))
som_2 = spn.SpiceNetSom(n_neurons=som_2_size,
                        value_range_start=df['result'].min(),
                        value_range_end=df['result'].max(),
                        lrf_tuning_curve=spn.ConstLRF(0.7),
                        lrf_interaction_kernel=spn.ConstLRF(0.7))

correlation_matrix = spn.SpiceNetHcm(som_1, som_2, spn.ConstLRF(0.7), spn.ConstLRF(0.7))

spice_net = spn.SpiceNet(correlation_matrix)

## Fitting / Training
Try to play around with the ```epochs_on_batch``` and ```batch_size``` parameter. Depending on the amount of data you have available more epochs can improve or falsify your network. 

In [4]:
spice_net.fit(df['init'].tolist(), df['result'].tolist(), 10, 100, print_output=True)

100%|██████████| 30/30 [00:14<00:00,  2.01it/s]

Time spend on the Components: 
Som: 9.917815685272217 s | Convolution Matrix: 5.008063316345215 s


## Plotting
The package provides you with some plotting functions. Each of them starts with plot as name. Feel free to expirement with them.

In [5]:
spn.plot_som(som_1, True)
spn.plot_som(som_2, True)
spn.plot_hcm(correlation_matrix)

## NIR Export
We first export ALL neurons, SOMs, HCM, and SPICEnet from the original implementation to NIR via the to_nir function

In [6]:
from spice_net.utility import nir_utils

exported_nir_neuron_som1 = [nir_utils.to_nir(neuron) for neuron in som_1.neurons]
exported_nir_neuron_som2 = [nir_utils.to_nir(neuron) for neuron in som_2.neurons]

exported_nir_som_1 = nir_utils.to_nir(som_1)
exported_nir_som_2 = nir_utils.to_nir(som_2)

exported_nir_hcm = nir_utils.to_nir(correlation_matrix)

exported_nir_spice_net = nir_utils.to_nir(spice_net)

## SINABS Import

Now we import all of these exported values to SINABS

In [7]:
sinabs_som_1_neurons = [sinabs.nir.from_nir(node, num_timesteps=100) for node in exported_nir_neuron_som1]
sinabs_som_2_neurons = [sinabs.nir.from_nir(node, num_timesteps=100) for node in exported_nir_neuron_som2]

sinabs_som_1 = sinabs.nir.from_nir(exported_nir_som_1, num_timesteps=100)
sinabs_som_2 = sinabs.nir.from_nir(exported_nir_som_2, num_timesteps=100)

sinabs_hcm = sinabs.nir.from_nir(exported_nir_hcm, num_timesteps=100)

sinabs_spice_net = sinabs.nir.from_nir(exported_nir_spice_net, num_timesteps=100)


# SINABS Export

Now we reexport from SINABS to NIR

In [8]:
import torch

exported_sinabs_som_1_neurons = [sinabs.nir.to_nir(node.spicenetsomneuron, sample_data=torch.Tensor([1])) for node in sinabs_som_1_neurons]
exported_sinabs_som_2_neurons = [sinabs.nir.to_nir(node.spicenetsomneuron, sample_data=torch.Tensor([1])) for node in sinabs_som_2_neurons]

exported_sinabs_som_1 = sinabs.nir.to_nir(sinabs_som_1.spicenetsom, sample_data=torch.Tensor([1]))
exported_sinabs_som_2 = sinabs.nir.to_nir(sinabs_som_2.spicenetsom, sample_data=torch.Tensor([1]))

exported_sinabs_hcm = sinabs.nir.to_nir(sinabs_hcm.spicenethcm, sample_data=torch.from_numpy(np.array([[som_1.get_activation_vector(df["init"][0]), som_2.get_activation_vector(df["result"][0])]])))

# SINABS export to NIR does a forward pass to confirm input node and output node of the NIR Graph
# Need to be careful! This changes the sinabs objects weights since the forward pass adjusts weights in the SPICEnet class!
# Another way to export the SPICEnet without changing weights is setting model.eval() before the export!
# This way the weights are not changed during the forward pass of the export
sinabs_spice_net.eval()
exported_sinabs_spice_net = sinabs.nir.to_nir(sinabs_spice_net.spicenet, sample_data=torch.Tensor([[1, 1, 1]]))

copy_weights = exported_sinabs_spice_net.nodes["model"].hcms["0_1"].weights.copy()
copy_act_bar_1 = exported_sinabs_spice_net.nodes["model"].hcms["0_1"].activation_bar_vector_1.copy()
copy_act_bar_2 = exported_sinabs_spice_net.nodes["model"].hcms["0_1"].activation_bar_vector_2.copy()

# As another work around we safe the last weights in the sinabs objects so that the export really gets the weights it wants!
# We only set it back to train here to visualise the difference between the two exports
sinabs_spice_net.train()
exported_sinabs_spice_net = sinabs.nir.to_nir(sinabs_spice_net.spicenet, sample_data=torch.Tensor([[1, 1, 1]]))

# The first test is still correct due to saving the last weights for the export
assert np.all(copy_weights == exported_sinabs_spice_net.nodes["model"].hcms["0_1"].weights)
assert np.all(copy_act_bar_1 == exported_sinabs_spice_net.nodes["model"].hcms["0_1"].activation_bar_vector_1)
assert np.all(copy_act_bar_2 == exported_sinabs_spice_net.nodes["model"].hcms["0_1"].activation_bar_vector_2)
print("Passed eval() test - not copying again since the same")

# Doing another, however, causes the weights to be wrong since the last weights are now the weights after the previous forward pass since another one happens here
exported_sinabs_spice_net_visualise_error = sinabs.nir.to_nir(sinabs_spice_net.spicenet, sample_data=torch.Tensor([[1, 1, 1]]))

# This is supposed to fail! Since we went into train mode and exported twice the first export caused a forward pass and adjusted the weights!
try:
    assert np.all(copy_weights == exported_sinabs_spice_net_visualise_error.nodes["model"].hcms["0_1"].weights)
    assert np.all(copy_act_bar_1 == exported_sinabs_spice_net_visualise_error.nodes["model"].hcms["0_1"].activation_bar_vector_1)
    assert np.all(copy_act_bar_2 == exported_sinabs_spice_net_visualise_error.nodes["model"].hcms["0_1"].activation_bar_vector_2)
except AssertionError:
    print("This is supposed to fail! Since we went into train mode and exported twice the first export caused a forward pass and adjusted the weights!")
    print("Should only export in eval() mode to avoid this issue!")


Passed eval() test - not copying again since the same
This is supposed to fail! Since we went into train mode and exported twice the first export caused a forward pass and adjusted the weights!
Should only export in eval() mode to avoid this issue!


In [9]:
exported_sinabs_som_1

NIRGraph(nodes={'input': Input(input_type={'input': array([], dtype=int64)}, metadata={}), 'model': SPICEnetSOM(neurons=[SPICEnetSOMNeuron(std=array(0.0056501), mean=array(-0.97720826), input_type={'input': array(0)}, output_type={'output': array(0)}, metadata={}), SPICEnetSOMNeuron(std=array(0.01224173), mean=array(-0.97002057), input_type={'input': array(0)}, output_type={'output': array(0)}, metadata={}), SPICEnetSOMNeuron(std=array(0.01318163), mean=array(-0.95879918), input_type={'input': array(0)}, output_type={'output': array(0)}, metadata={}), SPICEnetSOMNeuron(std=array(0.01396917), mean=array(-0.92356813), input_type={'input': array(0)}, output_type={'output': array(0)}, metadata={}), SPICEnetSOMNeuron(std=array(0.00959851), mean=array(-0.91275545), input_type={'input': array(0)}, output_type={'output': array(0)}, metadata={}), SPICEnetSOMNeuron(std=array(0.01149296), mean=array(-0.89470916), input_type={'input': array(0)}, output_type={'output': array(0)}, metadata={}), SPIC

# Import to SPICEnet Python Framework
And now we re-import the modules to the original Python Implementation and compare to the original objects

In [10]:
# Explicitly all the Learning rate functions are inferred!
imported_som_1_neurons = [nir_utils.from_nir(node) for node in exported_sinabs_som_1_neurons]
imported_som_2_neurons = [nir_utils.from_nir(node) for node in exported_sinabs_som_2_neurons]

imported_som_1 = nir_utils.from_nir(exported_sinabs_som_1)
imported_som_2 = nir_utils.from_nir(exported_sinabs_som_2)

imported_hcm = nir_utils.from_nir(exported_sinabs_hcm, som_1=imported_som_1, som_2=imported_som_2)

# Import Whole SpiceNet
imported_spice_net = nir_utils.from_nir(exported_sinabs_spice_net)

# Compare to Original
Now we compare the objects parameters to the original objects to see if any loss / error occured

In [11]:
# Test neurons
for i in range(len(imported_som_1_neurons)):
    assert imported_som_1_neurons[i].preferred_value == som_1.neurons[i].preferred_value
    assert imported_som_1_neurons[i].tuning_curve_width == som_1.neurons[i].tuning_curve_width

for i in range(len(imported_som_2_neurons)):
    assert imported_som_2_neurons[i].preferred_value == som_2.neurons[i].preferred_value
    assert imported_som_2_neurons[i].tuning_curve_width == som_2.neurons[i].tuning_curve_width

In [12]:
# Test SOMs
for i in range(len(imported_som_1.neurons)):    
    assert imported_som_1.neurons[i].preferred_value == som_1.neurons[i].preferred_value
    assert imported_som_1.neurons[i].tuning_curve_width == som_1.neurons[i].tuning_curve_width
    
for i in range(len(imported_som_2.neurons)):
    assert imported_som_2.neurons[i].preferred_value == som_2.neurons[i].preferred_value
    assert imported_som_2.neurons[i].tuning_curve_width == som_2.neurons[i].tuning_curve_width
    
assert imported_som_1.get_iteration() == som_1.get_iteration()
assert imported_som_2.get_iteration() == som_2.get_iteration()

assert imported_som_1.get_lrf_tuning_curve().get_parameters() == som_1.get_lrf_tuning_curve().get_parameters()
assert imported_som_1.get_lrf_interaction_kernel().get_parameters() == som_1.get_lrf_interaction_kernel().get_parameters()

assert imported_som_2.get_lrf_tuning_curve().get_parameters() == som_2.get_lrf_tuning_curve().get_parameters()
assert imported_som_2.get_lrf_interaction_kernel().get_parameters() == som_2.get_lrf_interaction_kernel().get_parameters()

In [13]:
# Test HCM
for i in range(som_size):
    for j in range(som_2_size):
        assert imported_hcm.weights[i, j] == correlation_matrix.weights[i, j]

for i in range(som_size):
    assert imported_hcm.activation_bar_vector_1[i] == correlation_matrix.activation_bar_vector_1[i]
    assert imported_hcm.activation_bar_vector_2[i] == correlation_matrix.activation_bar_vector_2[i]
    
assert imported_hcm.get_iteration() == correlation_matrix.get_iteration()
assert imported_hcm.get_trust_of_new_lrf().get_parameters() == correlation_matrix.get_trust_of_new_lrf().get_parameters()
assert imported_hcm.get_weights_lrf().get_parameters() == correlation_matrix.get_weights_lrf().get_parameters()
assert imported_hcm.get_trust_of_new_lrf().__class__.__name__ == correlation_matrix.get_trust_of_new_lrf().__class__.__name__
assert imported_hcm.get_weights_lrf().__class__.__name__ == correlation_matrix.get_weights_lrf().__class__.__name__

In [14]:
# Test SpiceNet
for i in range(som_size):
    for j in range(som_2_size):
        print(imported_spice_net.get_correlation_matrix().weights[i, j])
        print(correlation_matrix.weights[i, j])
        assert imported_spice_net.get_correlation_matrix().weights[i, j] == correlation_matrix.weights[i, j]

print(imported_spice_net.get_correlation_matrix().activation_bar_vector_1)
print(correlation_matrix.activation_bar_vector_1)

for i in range(som_size):
    assert imported_spice_net.get_correlation_matrix().activation_bar_vector_1[i] == correlation_matrix.activation_bar_vector_1[i]
    assert imported_spice_net.get_correlation_matrix().activation_bar_vector_2[i] == correlation_matrix.activation_bar_vector_2[i]
    assert imported_spice_net.get_som_1().neurons[i].preferred_value == som_1.neurons[i].preferred_value
    assert imported_spice_net.get_som_1().neurons[i].tuning_curve_width == som_1.neurons[i].tuning_curve_width
    assert imported_spice_net.get_som_2().neurons[i].preferred_value == som_2.neurons[i].preferred_value
    assert imported_spice_net.get_som_2().neurons[i].tuning_curve_width == som_2.neurons[i].tuning_curve_width
    
for i in range(som_size):
    assert imported_spice_net.get_correlation_matrix().activation_bar_vector_1[i] == correlation_matrix.activation_bar_vector_1[i]
    assert imported_spice_net.get_correlation_matrix().activation_bar_vector_2[i] == correlation_matrix.activation_bar_vector_2[i]
    assert imported_spice_net.get_som_1().neurons[i].preferred_value == som_1.neurons[i].preferred_value
    assert imported_spice_net.get_som_1().neurons[i].tuning_curve_width == som_1.neurons[i].tuning_curve_width
    assert imported_spice_net.get_som_2().neurons[i].preferred_value == som_2.neurons[i].preferred_value
    assert imported_spice_net.get_som_2().neurons[i].tuning_curve_width == som_2.neurons[i].tuning_curve_width

    assert imported_spice_net.get_correlation_matrix().weights.shape == correlation_matrix.weights.shape

    assert imported_spice_net.get_correlation_matrix().get_iteration() == correlation_matrix.get_iteration()
    assert imported_spice_net.get_correlation_matrix().get_trust_of_new_lrf().get_parameters() == correlation_matrix.get_trust_of_new_lrf().get_parameters()
    assert imported_spice_net.get_correlation_matrix().get_weights_lrf().get_parameters() == correlation_matrix.get_weights_lrf().get_parameters()
    assert imported_spice_net.get_correlation_matrix().get_trust_of_new_lrf().__class__.__name__ == correlation_matrix.get_trust_of_new_lrf().__class__.__name__
    assert imported_spice_net.get_correlation_matrix().get_weights_lrf().__class__.__name__ == correlation_matrix.get_weights_lrf().__class__.__name__

    assert imported_spice_net.som_1.get_iteration() == som_1.get_iteration()
    assert imported_spice_net.som_2.get_iteration() == som_2.get_iteration()

    assert imported_spice_net.som_1.get_lrf_tuning_curve().get_parameters() == som_1.get_lrf_tuning_curve().get_parameters()
    assert imported_spice_net.som_1.get_lrf_interaction_kernel().get_parameters() == som_1.get_lrf_interaction_kernel().get_parameters()

    assert imported_spice_net.som_2.get_lrf_tuning_curve().get_parameters() == som_2.get_lrf_tuning_curve().get_parameters()
    assert imported_spice_net.som_2.get_lrf_interaction_kernel().get_parameters() == som_2.get_lrf_interaction_kernel().get_parameters()

199307.536152701
199307.536152701
132575.47828335396
132575.47828335396
91660.19865857823
91660.19865857823
64736.76412824545
64736.76412824545
143624.82248041336
143624.82248041336
50357.48284986817
50357.48284986817
16229.023687339348
16229.023687339348
-443.9389903719927
-443.9389903719927
-3799.118288023103
-3799.118288023103
-5127.124590061785
-5127.124590061785
-2616.522511336641
-2616.522511336641
-1209.4108439437557
-1209.4108439437557
-792.7571114029911
-792.7571114029911
-757.3607572273601
-757.3607572273601
-1381.6433316243201
-1381.6433316243201
-1550.6294355034206
-1550.6294355034206
-149.58793904448146
-149.58793904448146
-1763.3186163999765
-1763.3186163999765
-6420.225072225021
-6420.225072225021
-660.9827143178638
-660.9827143178638
-1294.8679302288106
-1294.8679302288106
-1015.087268383735
-1015.087268383735
-629.2005692914662
-629.2005692914662
-1533.6020855808936
-1533.6020855808936
-1601.8219850486198
-1601.8219850486198
-1259.1403035374788
-1259.1403035374788
-121

## Decoding
A this point in time only the naive decoding mechanism is implemented. It is straight forward just call the decode method.

In [15]:
test_data = df.sample()
inti_test_value = test_data['init'].iloc[0]
result_test_value = test_data['result'].iloc[0]

In [16]:
spice_net_som_1_activations = spice_net.som_1.get_activation_vector(inti_test_value)

In [17]:
nir_spice_net_som_1_activations = imported_spice_net.som_1.get_activation_vector(inti_test_value)

In [18]:
assert np.all(spice_net_som_1_activations == nir_spice_net_som_1_activations)

In [19]:
som_2_should_activation = spice_net.get_correlation_matrix().calculate_som_1_to_2(spice_net_som_1_activations)

In [20]:
nir_som_2_should_activation = imported_spice_net.get_correlation_matrix().calculate_som_1_to_2(nir_spice_net_som_1_activations)

In [21]:
assert np.all(som_2_should_activation == nir_som_2_should_activation)

In [22]:
test_data = df.sample()
print(f'Init: {inti_test_value}, Result: {result_test_value}')
print(f'Predicted: {spice_net.decode(inti_test_value)}')
print(f"Predicted imported: {imported_spice_net.decode(inti_test_value)}")

Init: 0.1882448367342717, Result: 0.0066706663522799
Predicted: 0.008447602731830272
Predicted imported: 0.008447602731830272


In [23]:
errors = []

for i in range(100):
    test_data = df.sample()
    inti_test_value = test_data['init'].iloc[0]
    result_test_value = test_data['result'].iloc[0]
    try:
        predicted = spice_net.decode(inti_test_value)
    except Exception as e:
        print("Error in original spicenet, continuing..")
        continue
    predicted_imported = imported_spice_net.decode(inti_test_value)
    assert predicted == predicted_imported
    errors.append(abs(predicted - result_test_value) / 2 * 100)
    print(f'Error: {errors[-1]:.6f},\t Predicted: {predicted:.6f},\t Predicted Imported: {predicted_imported:.6f} Actual: {result_test_value:.6f}')

print(f'Mean Error in % {np.mean(np.array(errors))}')
print(f'Median Error in % {np.median(np.array(errors))}')
    

Error: 0.395447,	 Predicted: -0.533201,	 Predicted Imported: -0.533201 Actual: -0.525292
Error: 4.193722,	 Predicted: -0.998941,	 Predicted Imported: -0.998941 Actual: -0.915066
Error: 0.198389,	 Predicted: -0.122676,	 Predicted Imported: -0.122676 Actual: -0.118708
Error: 1.323119,	 Predicted: 0.152545,	 Predicted Imported: 0.152545 Actual: 0.179007
Error: 0.418665,	 Predicted: -0.191265,	 Predicted Imported: -0.191265 Actual: -0.199639
Error: 4.089581,	 Predicted: -0.312618,	 Predicted Imported: -0.312618 Actual: -0.394410
Error: 0.924108,	 Predicted: 0.851431,	 Predicted Imported: 0.851431 Actual: 0.832949
Error: 0.336228,	 Predicted: 0.009123,	 Predicted Imported: 0.009123 Actual: 0.002399
Error: 0.728282,	 Predicted: -0.042732,	 Predicted Imported: -0.042732 Actual: -0.028166
Error: 1.046956,	 Predicted: 0.525116,	 Predicted Imported: 0.525116 Actual: 0.546055
Error: 0.431601,	 Predicted: 0.017667,	 Predicted Imported: 0.017667 Actual: 0.009035
Error: 0.993977,	 Predicted: 0.92052